# dataeuropa: worked examples (Julia)

Three tasks with [data.europa.eu](https://data.europa.eu), the European open data portal, from a
search to a result: air quality in Lisbon from a CSV file, who publishes air-quality data and in
what formats, and the boundaries of Gelsenkirchen from a feature service.

The same examples, with explanations, are in the [Julia documentation](https://randsley.github.io/dataeuropa/julia/examples/); notebooks for the
other two languages are next to this one. Install: `]add DataEuropa DataFrames GeoDataFrames` (DataEuropa is being registered in General; until then `]add https://github.com/randsley/dataeuropa:julia`).

The outputs were stored when the notebook was last run (the date is printed below each data
source). The portal changes all the time, so counts and search results will differ when you run it.

In [1]:
using DataEuropa, DataFrames, Statistics, Dates

println("DataEuropa ", pkgversion(DataEuropa))
retrieved = string(today());

DataEuropa 0.1.1


## 1. Air quality in Lisbon: from a search to a summary table

Find air-quality data from Portugal that comes as CSV. Quotes make the search match the phrase,
not the separate words.

In [2]:
hits = search_datasets("\"air quality\""; country="pt", format="CSV")
hits[:, [:id, :title, :publisher]]

Row,id,title,publisher
,String,String,String
1,67a5e928a3ee7360b61cd5e4,"Sensing: Monthly reports with Air Quality, Sound Pressure and Weather data",Câmara Municipal de Oeiras
2,6689f1686e4bd9bc98ed011b,Monitoring of Environmental Parameters of the City of Lisbon - Historical 15/07/2021 to 15/01/2024,Município de Lisboa


The Lisbon dataset has one file per period: `QA` files hold air quality, `ME` files weather. List
the files only (no services or web pages), then read the first quarter of 2024. The file's
checksum is checked when the portal has one.

In [3]:
files = distributions("6689f1686e4bd9bc98ed011b"; files_only=true)
files[:, [:title, :format]]

Row,title,format
,String,String
1,ME_2024_S1,CSV
2,QA_2023,ZIP
3,ME_2022,ZIP
4,QA_2022,ZIP
5,ME_2021_S2,CSV
6,QA_2024_Q1,CSV
7,ME_2023,ZIP
8,Traffic,CSV
9,Noise,CSV


In [4]:
row = files[findfirst(isequal("QA_2024_Q1"), files.title), :]
qa = load_distribution(row)
first(qa[:, [:DTM_LOCAL, :PARAMETRO, :LOCAL, :UNIDADE, :VALOR]], 3)

Row,DTM_LOCAL,PARAMETRO,LOCAL,UNIDADE,VALOR
,String,String,String,String,Float64
1,2024-01-05 19:00:00.0,PM25,Calçada da Ajuda,µg/m3,6.0
2,2024-01-06 03:00:00.0,PM25,Calçada da Ajuda,µg/m3,5.0
3,2024-01-06 04:00:00.0,PM25,Calçada da Ajuda,µg/m3,3.0


The file is in long format: one row per reading, with the pollutant in `PARAMETRO`. `-99` marks a
missing reading. Some stations have only a few readings, so count them too and keep the stations
with at least 300 before comparing NO₂ means.

In [5]:
no2 = filter(r -> r.PARAMETRO == "NO2" && coalesce(r.VALOR >= 0, false), qa)
by_station = combine(groupby(no2, :LOCAL), :VALOR => (v -> round(mean(v); digits=1)) => :mean, nrow => :count)
first(sort(filter(:count => >=(300), by_station), :mean; rev=true), 5)

Row,LOCAL,mean,count
,String,Float64,Int64
1,Avenida António Augusto de Aguiar,111.9,356
2,Campo de Ourique,95.3,356
3,Praça São Francisco de Assis,93.9,356
4,Rua Morais Soares,92.0,356
5,Estrada de Monsanto,78.0,356


In [6]:
lisbon = get_dataset("6689f1686e4bd9bc98ed011b")
println("Source:    ", lisbon.title)
println("Publisher: ", lisbon.publisher)
println("Portal:    ", lisbon.portal_url)
println("File:      ", row.title, " (first quarter of 2024), ", row.url)
println("Licence:   ", coalesce(row.license, "not stated on the portal"))
println("Retrieved: ", retrieved)

Source:    Monitoring of Environmental Parameters of the City of Lisbon - Historical 15/07/2021 to 15/01/2024
Publisher: Município de Lisboa
Portal:    https://data.europa.eu/data/datasets/6689f1686e4bd9bc98ed011b
File:      QA_2024_Q1 (first quarter of 2024), https://dados.cm-lisboa.pt/dataset/9954113c-3f2c-44ea-a59c-5b989da83156/resource/c580a5a0-d2fc-458c-921b-9a08dbc2893c/download/qa2024q1.csv
Licence:   https://creativecommons.org/licenses/by/4.0/
Retrieved: 2026-11-28


## 2. Who publishes air-quality data, and in what formats

Counts and facets need no records, so they are fast. `EUROPE` is the value for datasets about
Europe as a whole. A dataset with several formats counts once for each.

In [7]:
count_datasets("\"air quality\"")

3582

In [8]:
facet_values("country", "\"air quality\""; top=8)

Row,id,label,count
,String,String,Int64
1,EUROPE,Europe,621
2,de,Germany,539
3,fr,France,449
4,gb,United Kingdom,373
5,es,Spain,347
6,it,Italy,247
7,ro,Romania,189
8,bg,Bulgaria,147


In [9]:
facet_values("format", "\"air quality\""; top=6)

Row,id,label,count
,String,String,Int64
1,CSV,CSV,862
2,JSON,JSON,536
3,XML,XML,391
4,WMS_SRVC,WMS,244
5,ZIP,ZIP,166
6,HTML,HTML,147


In [10]:
println("Source: data.europa.eu Search API (counts and facets), retrieved ", retrieved)

Source: data.europa.eu Search API (counts and facets), retrieved 2026-11-28


## 3. Boundaries of Gelsenkirchen from a feature service

The city of Gelsenkirchen publishes its administrative boundaries as a GeoServer service, which the
portal lists as OGC API Features and as WFS. This needs GeoDataFrames.jl (`using GeoDataFrames` loads the geodata extension).

The OGC API Features entry points to one collection, the cadastral districts, in ETRS89 / UTM zone
32N (EPSG:25832, metres), so areas can be computed directly. The WFS entry is the whole service,
with 22 layers; pick the city quarters by name (the namespace prefix can be left out) and reproject
them to longitude and latitude.

In [11]:
using GeoDataFrames

dists = distributions("eac1e504-9d63-4670-95b3-3ef04e22aa6e~~1")
dists[:, [:format, :kind]]

Row,format,kind
,String,String
1,OGCFEAT,service
2,JSON,file
3,WFS_SRVC,service


In [12]:
districts = load_geodata(dists[findfirst(isequal("OGCFEAT"), dists.format), :])
districts.area_km2 = round.(GeoDataFrames.AG.geomarea.(districts.geometry) ./ 1e6; digits=2)
first(sort(districts[:, [:gemarkung_name, :area_km2]], :area_km2; rev=true), 3)

Row,gemarkung_name,area_km2
,String,Float64
1,Buer,61.29
2,Bismarck,10.08
3,Heßler,6.88


In [13]:
quarters = load_geodata(dists[findfirst(isequal("WFS_SRVC"), dists.format), :]; layer="stadtteil", crs="EPSG:4326")
println(nrow(quarters), " city quarters")
first(quarters[:, Not(:geometry)], 3)

18 city quarters


Row,gml_id,stadtteil_id,stadtteil_name,stadtbezirk_id,stadtbezirk_name,stadtteil_fläche
,String,Int32,String,Int32,String,Float64
1,stadtteil.1,10,Altstadt,1,Mitte,1.58741e6
2,stadtteil.2,11,Schalke,1,Mitte,3.04087e6
3,stadtteil.3,12,Schalke-Nord,1,Mitte,4.27994e6


In [14]:
gk = get_dataset("eac1e504-9d63-4670-95b3-3ef04e22aa6e~~1")
licences = sort(unique(skipmissing(dists.license)))
println("Source:    ", gk.title, " (catalogue ", gk.catalog, ")")
println("Publisher: ", coalesce(gk.publisher, "not named in the record; contact point " * gk.raw.contact_point[1].url[1]))
println("Portal:    ", gk.portal_url)
println("Licence:   ", isempty(licences) ? "not stated on the portal (rights: no access restrictions)" : join(licences, ", "))
println("Retrieved: ", retrieved)

Source:    Municipalities of Gelsenkirchen (catalogue gdi-de)
Publisher: not named in the record; contact point https://www.gelsenkirchen.de
Portal:    https://data.europa.eu/data/datasets/eac1e504-9d63-4670-95b3-3ef04e22aa6e~~1
Licence:   not stated on the portal (rights: no access restrictions)
Retrieved: 2026-11-28
